In [1]:
import subprocess, sys
subprocess.run([sys.executable,"-m","pip","install",
    "unsloth[kaggle-new]","--upgrade","--quiet"], check=True)
subprocess.run([sys.executable,"-m","pip","install",
    "unsloth_zoo","hf_transfer","--upgrade","--quiet"], check=True)

import torch
print("GPU: " + str(torch.cuda.get_device_name(0)))
print("Cell 1 done")

GPU: Tesla T4
Cell 1 done


In [2]:
import os
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, HfApi

# ─── Only line you change ──────────────────────
HF_USERNAME   = "IbadUrRahman"
ADAPTER_REPO  = HF_USERNAME + "/omnimind-llama3-adapter"
MERGED_REPO   = HF_USERNAME + "/omnimind-llama3-merged"
BASE_MODEL    = "meta-llama/Meta-Llama-3.1-8B-Instruct"
MAX_SEQ_LEN   = 1024
# ───────────────────────────────────────────────

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN, add_to_git_credential=False)

api = HfApi(token=HF_TOKEN)
try:
    api.create_repo(repo_id=MERGED_REPO, private=True, exist_ok=True)
    print("Merged repo ready: https://huggingface.co/" + MERGED_REPO)
except Exception as e:
    print("Repo note: " + str(e))

print("Auth complete")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Merged repo ready: https://huggingface.co/IbadUrRahman/omnimind-llama3-merged
Auth complete


In [3]:
import torch
import os
from unsloth import FastLanguageModel

print("Step 1: Loading base model (4-bit)...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = BASE_MODEL,
    max_seq_length = MAX_SEQ_LEN,
    dtype          = None,
    load_in_4bit   = True,
    token          = HF_TOKEN,
)
print("Base model loaded")

print("Step 2: Loading LoRA adapters from HF Hub...")
# FIXED: Using Unsloth's native adapter loader
model.load_adapter(ADAPTER_REPO)
print("Adapters loaded")

print("Step 3 & 4: Merging and saving locally via Unsloth...")
print("  (This takes 5-10 minutes — do not interrupt)")
SAVE_DIR = "/kaggle/working/merged_model"

# FIXED: Unsloth's built-in save handles the 4-bit unwrapping and merging
model.save_pretrained_merged(
    SAVE_DIR, 
    tokenizer, 
    save_method = "merged_16bit",
)
print("Merged model saved locally")

files = os.listdir(SAVE_DIR)
for f in sorted(files):
    size = os.path.getsize(SAVE_DIR + "/" + f) / 1024**3
    print("  " + f + "  (" + str(round(size,2)) + " GB)")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Step 1: Loading base model (4-bit)...
==((====))==  Unsloth 2026.5.9: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/meta-llama-3.1-8b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


Base model loaded
Step 2: Loading LoRA adapters from HF Hub...


Loading weights:   0%|          | 0/448 [00:00<?, ?it/s]

Adapters loaded
Step 3 & 4: Merging and saving locally via Unsloth...
  (This takes 5-10 minutes — do not interrupt)
Unsloth: Saving full fine-tuned model to '/kaggle/working/merged_model' ...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/merged_model/tokenizer_config.json.


Unsloth: Model saved successfully to '/kaggle/working/merged_model'
Merged model saved locally
  adapter_config.json  (0.0 GB)
  adapter_model.safetensors  (0.15 GB)
  chat_template.jinja  (0.0 GB)
  config.json  (0.0 GB)
  generation_config.json  (0.0 GB)
  tokenizer.json  (0.02 GB)
  tokenizer_config.json  (0.0 GB)


In [4]:
from huggingface_hub import HfApi

api = HfApi(token=HF_TOKEN)
print("Uploading merged model to HF Hub (~16GB)...")
print("This takes 15-25 minutes depending on Kaggle upload speed")

api.upload_folder(
    folder_path    = "/kaggle/working/merged_model",
    repo_id        = MERGED_REPO,
    repo_type      = "model",
    commit_message = "Full merged model: LLaMA 3.1-8B + OmniMind LoRA (11 epochs)",
    token          = HF_TOKEN,
)

print("")
print("=" * 55)
print("  MERGE COMPLETE")
print("=" * 55)
print("  Merged model: https://huggingface.co/" + MERGED_REPO)
print("  This model is now standalone — no adapter needed")
print("  Proceed to deployment step")
print("=" * 55)

Uploading merged model to HF Hub (~16GB)...
This takes 15-25 minutes depending on Kaggle upload speed


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            


  MERGE COMPLETE
  Merged model: https://huggingface.co/IbadUrRahman/omnimind-llama3-merged
  This model is now standalone — no adapter needed
  Proceed to deployment step
